In [1]:
import torch
import torch.nn as nn

In [2]:
import pandas as pd

In [3]:
print(torch.cuda.is_available())
device= torch.device("cuda")

True


In [4]:
df=pd.read_csv(r"C:\Users\xhu70\Projects\twel_data_collection\data\sensor_data.csv")
df.head()

,timestamp,temperature_c,humidity_pct,pressure_hpa
0,2026-03-07T18:12:00,27.17,27.88,1012.39
1,2026-03-07T18:12:10,27.16,27.64,1012.42
2,2026-03-07T18:12:20,27.16,27.53,1012.41
3,2026-03-07T18:12:30,27.15,27.39,1012.39
4,2026-03-07T18:12:40,27.15,27.76,1012.44


In [5]:
features=["temperature_c","humidity_pct","pressure_hpa"]
X_np= df[features].values
X_np

array([[  27.17,   27.88, 1012.39],
       [  27.16,   27.64, 1012.42],
       [  27.16,   27.53, 1012.41],
       ...,
       [  23.4 ,   53.97, 1019.29],
       [  23.39,   53.93, 1019.28],
       [  23.4 ,   53.96, 1019.29]], shape=(25000, 3))

In [6]:
def make_windows(data, window_size):
    X,y=[],[]
    for i in range(len(data)-window_size):
        X.append(data[i:i+window_size])
        y.append(data[i+window_size,0])
    
    return torch.tensor(X,dtype=torch.float32), torch.tensor(y,dtype=torch.float32)

In [7]:
window_size=60  # 10min
X,y=make_windows(X_np, window_size)

C:\Users\xhu70\AppData\Local\Temp\ipykernel_1756\2961653156.py:7: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  return torch.tensor(X,dtype=torch.float32), torch.tensor(y,dtype=torch.float32)


In [8]:
split=int(len(X)*0.8)
X_train,X_test=X[:split],X[split:]
y_train,y_test=y[:split],y[split:]


In [9]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)



In [10]:
import torch.nn as nn
import torch
import math
import torch.nn.functional as F

In [11]:
class TransformerForecast(nn.Module):
    def __init__(self, input_dim,d_model, num_heads, num_layers):
        super().__init__()
        self.embedding=nn.Linear(input_dim, d_model)
        self.d_model=d_model
        encoder_layer=nn.TransformerEncoderLayer(d_model=d_model,nhead=num_heads, batch_first=True)
        self.transformer=nn.TransformerEncoder(encoder_layer,num_layers=num_layers)
        self.fc=nn.Linear(d_model,1)


    def get_positioning_encoding(self, seq_len, d_model):
        pe= torch.zeros(seq_len,d_model)
        position=torch.arange(0,seq_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * 
                            (-math.log(10000.0) / d_model))
        pe[:,0::2]=torch.sin(position*div_term)
        pe[:,1::2]=torch.cos(position*div_term)
        return pe.to(device)
    
    def forward(self, x):
        out=self.embedding(x)
        pe=self.get_positioning_encoding(len(x[0]),self.d_model)
        out=out+pe
        out=self.transformer(out)
        out=out[:,-1,:]
        out=self.fc(out)   #[32, 1]
        out=out.squeeze(-1)

        #print(out.shape)
        return out

    
model=TransformerForecast(input_dim=3, d_model=12, num_heads=4, num_layers=2).to(device)
batch_x,batch_y= next(iter(train_loader))
batch_x = batch_x.to(device) 
batch_y = batch_y.to(device) 

out= model(batch_x)

In [12]:
import torch.optim as optim
criterion=nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
num_epochs=20

for epoch in range(num_epochs):
    model.train()
    total_loss=0

    for batch_x, batch_y in train_loader:
        batch_x=batch_x.to(device)
        batch_y=batch_y.to(device)
        optimizer.zero_grad()
        pred=model(batch_x)
        loss=criterion(pred, batch_y)
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
    
    avg_loss= total_loss/ len(train_loader)
    print(epoch+1, avg_loss)

1 280.4872057774128
2 26.8863182977224
3 22.94416700876676
4 8.704427759807844
5 3.649410216472088
6 1.9811380155957663
7 1.1173708513617897
8 0.8986388155235312
9 0.7893891505395564
10 0.6735450643926668
11 0.6077064474423727
12 0.5277724093279968
13 0.5354104706277258
14 0.46287855045058024
15 0.40470979667197055
16 0.4006421263150584
17 0.3658422002544961
18 0.3425445524521936
19 0.2874818701141824
20 0.25774875945913106


In [13]:
model.eval()
preds, actuals=[],[]

with torch.no_grad():
    for batch_x,batch_y in test_loader:
        batch_x=batch_x.to(device)
        batch_y=batch_y.to(device)
        pred=model(batch_x)
        preds.append(pred.cpu())
        actuals.append(batch_y.cpu())

preds = torch.cat(preds)
actuals = torch.cat(actuals)

rmse = torch.sqrt(nn.MSELoss()(preds, actuals))
print(f"Test RMSE: {rmse:.4f}")


Test RMSE: 1.0000


In [14]:
print(df['temperature_c'].min(), df['temperature_c'].max())
print(df['temperature_c'].std())

18.77 49.89
4.8201432903244
